Import Libraries

In [ ]:
# =========================================
# IMPORT LIBRARIES
# =========================================
import pandas as pd

from pyomo.environ import (
    ConcreteModel, Set, Param, Var,
    Constraint, Objective,
    NonNegativeReals, Binary,
    SolverFactory, minimize, value
)

Read Data

In [ ]:
# =========================================
# READ DATA FROM EXCEL
# =========================================

file_path = r"D:\UGM\Tugas Akhir\UC Coding\Belajar\DataSet_R05.xlsx"   # ganti sesuai nama file Anda

# Generator data
gen_df = pd.read_excel(file_path, sheet_name="Gen")

# Demand data
demand_df = pd.read_excel(file_path, sheet_name="Demand")

# Tampilkan untuk cek
#print(gen_df.head())
#print(demand_df.head())

Data Frame

In [ ]:
# ============================
# CONVERT DATAFRAME → DICTIONARY
# ============================

# Set generator
G = gen_df['Unit'].tolist()

# Set time
T = demand_df['Hour'].tolist()

# Parameter generator
Pmin = dict(zip(gen_df['Unit'], gen_df['Pmin']))
Pmax = dict(zip(gen_df['Unit'], gen_df['Pmax']))
a = dict(zip(gen_df['Unit'], gen_df['a']))
b = dict(zip(gen_df['Unit'], gen_df['b']))
c = dict(zip(gen_df['Unit'], gen_df['c']))

SU_cost = dict(zip(gen_df['Unit'], gen_df['SU_Cost']))
SD_cost = dict(zip(gen_df['Unit'], gen_df['SD_Cost']))

RampUp = dict(zip(gen_df['Unit'], gen_df['Ramp_Up']))
RampDown = dict(zip(gen_df['Unit'], gen_df['Ramp_Down']))

MinUp = dict(zip(gen_df['Unit'], gen_df['Min_Up']))
MinDown = dict(zip(gen_df['Unit'], gen_df['Min_Down']))

# Demand
Demand = dict(zip(demand_df['Hour'], demand_df['Demand']))

# Frequency
Droop = dict(zip(gen_df['Unit'], gen_df['Droop']))
PFR_Fraction = dict(zip(gen_df['Unit'], gen_df['PFR_Fraction']))
FreeGovernor = dict(zip(gen_df['Unit'], gen_df['FreeGovernor']))

# Type
Type = dict(zip(gen_df['Unit'], gen_df['Type']))

# Renewable Potential
PV_Potential = dict(zip(demand_df['Hour'], demand_df['Potensi PV (MW)']))
WT_Potential = dict(zip(demand_df['Hour'], demand_df['Potensi WT (MW)']))

RenewPotential = {}

for g in G:
    for t in T:
        if Type[g] == "PV":
            RenewPotential[g, t] = PV_Potential[t]
        elif Type[g] == "WT":
            RenewPotential[g, t] = WT_Potential[t]
        else:
            RenewPotential[g, t] = Pmax[g]

# Parameter Turunan
PFR_Max = {g: PFR_Fraction[g] * Pmax[g] for g in gen_df['Unit']}

# Data Frekuensi
f0 = 50.0
#f_min = 49.5
df_max = 0.5

# ===============================
# Frequency Bias turunan dari Droop
# B_g = Pmax / (Droop * f0)
# satuan: MW/Hz
# ===============================

FreqBias = {}

for g in G:
    if Droop[g] > 0 and FreeGovernor[g] == 1:
        FreqBias[g] = Pmax[g] / (Droop[g] * f0)
    else:
        FreqBias[g] = 0

#print("FreqBias =", FreqBias)

H = dict(zip(gen_df['Unit'], gen_df['H']))
Tg = dict(zip(gen_df['Unit'], gen_df['Tg']))

Definisi Model + Sets

In [ ]:
model = ConcreteModel()

model.G = Set(initialize=G)
model.T = Set(initialize=T)

# Reserve Cost
ReserveCost = {g: 0.10 * b[g] for g in G}

# ============================
# RESERVE REQUIREMENT
# ============================
#ReserveReq = {t: 0.10 * Demand[t] for t in Demand}

# ============================
# CONTINGENCY-BASED RESERVE
# ============================
#Contingency = max(Pmax.values())   # kapasitas unit terbesar
#ReserveReq = {t: Contingency for t in Demand}

#print("Contingency adopted =", Contingency, "MW")

# kebutuhan total pfr
#PFRReq = {t: 0.3 * Contingency for t in T}
#print("PFR =",PFRReq,"??")


Parameter Pyomo

In [ ]:
model.Pmin = Param(model.G, initialize=Pmin)
model.Pmax = Param(model.G, initialize=Pmax)

model.a = Param(model.G, initialize=a)
model.b = Param(model.G, initialize=b)
model.c = Param(model.G, initialize=c)

model.SU_cost = Param(model.G, initialize=SU_cost)
model.SD_cost = Param(model.G, initialize=SD_cost)

model.RampUp = Param(model.G, initialize=RampUp)
model.RampDown = Param(model.G, initialize=RampDown)

model.MinUp = Param(model.G, initialize=MinUp)
model.MinDown = Param(model.G, initialize=MinDown)

model.Demand = Param(model.T, initialize=Demand)

#model.ReserveReq = Param(model.T, initialize=ReserveReq)

model.Droop = Param(model.G, initialize=Droop, within=NonNegativeReals)
model.PFR_Fraction = Param(model.G, initialize=PFR_Fraction, within=NonNegativeReals)
model.FreeGovernor = Param(model.G, initialize=FreeGovernor, within=Binary)
model.PFR_Max = Param(model.G, initialize=PFR_Max, within=NonNegativeReals)

#model.PFRReq = Param(model.T, initialize=PFRReq)

model.delta_f_max = Param(initialize=df_max)

model.K_freq = Param(initialize=1000)

model.FreqBias = Param(model.G, initialize=FreqBias)

model.ReserveCost = Param(model.G, initialize=ReserveCost)

model.RenewPotential = Param(model.G, model.T, initialize=RenewPotential, within=NonNegativeReals)

model.H = Param(model.G, initialize=H)
model.Tg = Param(model.G, initialize=Tg)

Variables

In [ ]:
# =====================
# Variables
# =====================
model.P = Var(model.G, model.T, domain=NonNegativeReals)
model.u = Var(model.G, model.T, domain=Binary)
model.y = Var(model.G, model.T, domain=Binary)              #startup
model.z = Var(model.G, model.T, within=Binary)              #shutdown
model.R = Var(model.G, model.T, within=NonNegativeReals)    #spinning reserve
model.R_pfr = Var(model.G, model.T, within=NonNegativeReals) #model PFR
model.delta_f = Var(model.T, within=NonNegativeReals)            #deviasi frekuensi
model.Curt = Var(model.G, model.T, within=NonNegativeReals) #curtailment
#model.Contingency = Var(model.T, within=NonNegativeReals)   #contingency

Objective Function

In [ ]:
# =========================================
# OBJECTIVE FUNCTION
# =========================================
def obj_rule(m):
    return sum(
        m.b[g] * m.P[g, t]              #Cost Koefisien b
        + m.c[g] * m.u[g, t]            #Cost Koefisien c
        + m.SU_cost[g] * m.y[g, t]      #Start Up Cost
        + m.SD_cost[g] * m.z[g, t]      #Shutdown Cost
        + m.ReserveCost[g] * m.R[g,t]   #Reserve Cost
        for g in m.G for t in m.T
    )

model.Obj = Objective(rule=obj_rule, sense=minimize)

Constraints

In [ ]:
# =====================
# Power Constraints
# =====================

# Power balance
def balance_rule(m, t):
    return sum(m.P[g, t] for g in m.G) == m.Demand[t]

model.Balance = Constraint(model.T, rule=balance_rule)

# Capacity upper bound
def max_rule(m, g, t):
    if Type[g] in ["PV", "WT"]:
        return m.P[g, t] <= m.Pmax[g] * m.u[g, t]
    else:
        return m.P[g, t] <= m.Pmax[g] * m.u[g, t]

model.MaxCap = Constraint(model.G, model.T, rule=max_rule)

# Capacity lower bound
def min_rule(m, g, t):
    return m.P[g, t] >= m.Pmin[g] * m.u[g, t]

model.MinCap = Constraint(model.G, model.T, rule=min_rule)

# Startup / Shutdown transition
def transition_rule(m, g, t):
    if t == 1:
        return m.u[g, t] == m.y[g, t] - m.z[g, t]
    else:
        return m.u[g, t] - m.u[g, t-1] == m.y[g, t] - m.z[g, t]

model.Transition = Constraint(model.G, model.T, rule=transition_rule)

# Ramp Up
def ramp_up_rule(m, g, t):
    if t == 1:
        return Constraint.Skip
    return m.P[g, t] - m.P[g, t-1] <= m.RampUp[g]

model.RampUpConstraint = Constraint(model.G, model.T, rule=ramp_up_rule)

# Ramp Down
def ramp_down_rule(m, g, t):
    if t == 1:
        return Constraint.Skip
    return m.P[g, t-1] - m.P[g, t] <= m.RampDown[g]

model.RampDownConstraint = Constraint(model.G, model.T, rule=ramp_down_rule)

# Minimum Up Time
def min_up_rule(m, g, t):
    MU = int(value(m.MinUp[g]))
    if t + MU - 1 > max(m.T):
        return Constraint.Skip
    return sum(m.u[g, k] for k in range(t, t + MU)) >= MU * m.y[g, t]

model.MinUpConstraint = Constraint(model.G, model.T, rule=min_up_rule)

# Minimum Down Time
def min_down_rule(m, g, t):
    MD = int(value(m.MinDown[g]))
    if t + MD - 1 > max(m.T):
        return Constraint.Skip
    return sum(1 - m.u[g, k] for k in range(t, t + MD)) >= MD * m.z[g, t]

model.MinDownConstraint = Constraint(model.G, model.T, rule=min_down_rule)

# Reserve per generator
def reserve_cap_rule(m, g, t):
    if Type[g] in ["PV", "WT"]:
        return m.R[g, t] == 0
    else:
        return m.P[g, t] + m.R[g, t] <= m.Pmax[g] * m.u[g, t]

model.ReserveCap = Constraint(model.G, model.T, rule=reserve_cap_rule)

# Reserve system
#def system_reserve_rule(m, t):
#    return sum(m.R[g, t] for g in m.G) >= m.Contingency[t]

#model.SystemReserve = Constraint(model.T, rule=system_reserve_rule)

#def system_reserve_rule(m, g, t):
#    if Type[g] not in ["PV", "WT"]:
#        return sum(m.R[k, t] for k in m.G) >= m.P[g, t]
#    else:
#        return Constraint.Skip

#model.SystemReserve = Constraint(model.G, model.T, rule=system_reserve_rule)


# N-1 reserve: reserve harus berasal dari unit lain, bukan unit yang trip
def n1_reserve_rule(m, outage_g, t):
    if Type[outage_g] not in ["PV", "WT"]:
        return sum(
            m.R[g, t]
            for g in m.G
            if g != outage_g and Type[g] not in ["PV", "WT"]
        ) >= m.P[outage_g, t]
    else:
        return Constraint.Skip

model.N1Reserve = Constraint(model.G, model.T, rule=n1_reserve_rule)

# Curtailment
def curtailment_rule(m, g, t):
    if Type[g] in ["PV", "WT"]:
        return m.Curt[g, t] == m.RenewPotential[g, t] - m.P[g, t]
    else:
        return m.Curt[g, t] == 0

model.Curtailment = Constraint(model.G, model.T, rule=curtailment_rule)

def renewable_limit_rule(m, g, t):
    if Type[g] in ["PV", "WT"]:
        return m.P[g, t] <= m.RenewPotential[g, t]
    else:
        return Constraint.Skip

model.RenewLimit = Constraint(model.G, model.T, rule=renewable_limit_rule)

#Contingency
#def contingency_rule(m, g, t):
#    if Type[g] not in ["PV", "WT"]:
#        return m.Contingency[t] >= m.P[g,t]
#    else:
#        return Constraint.Skip

#model.ContingencyLimit = Constraint(model.G, model.T, rule=contingency_rule)

In [ ]:
# =====================
# PFR Constraints
# =====================

use_pfr = False   # True = PFR scenario, False = No-PFR scenario

if use_pfr:

# PFR dibatasi headroom unit
    def pfr_headroom_rule(m, g, t):
        return m.R_pfr[g, t] <= m.Pmax[g] * m.u[g, t] - m.P[g, t]

    model.PFR_Headroom = Constraint(model.G, model.T, rule=pfr_headroom_rule)

# PFR dibatasi kemampuan unit dan status free governor
    def pfr_cap_rule(m, g, t):
        return m.R_pfr[g, t] <= m.PFR_Max[g] * m.FreeGovernor[g] * m.u[g, t]

    model.PFR_Cap = Constraint(model.G, model.T, rule=pfr_cap_rule)

# PFR harus bagian dari reserve total
    def pfr_within_reserve_rule(m, g, t):
        return m.R_pfr[g, t] <= m.R[g, t]

    model.PFR_Within_Reserve = Constraint(model.G, model.T, rule=pfr_within_reserve_rule)

    def total_pfr_rule(m, t):
        return sum(m.R_pfr[g, t] for g in m.G) >= m.PFRReq[t]

    model.TotalPFR = Constraint(model.T, rule=total_pfr_rule)

    #def freq_limit_rule(m, t):
    #    return m.delta_f[t] <= m.delta_f_max

    #model.FreqLimit = Constraint(model.T, rule=freq_limit_rule)

    #def freq_response_rule(m, t):
    #    return sum(m.R_pfr[g, t] / m.Droop[g] for g in m.G if m.Droop[g] > 0) >= m.delta_f[t] * m.K_freq

    #model.FreqResponse = Constraint(model.T, rule=freq_response_rule)

# ==========================================
# Frequency deviation linked to contingency and PFR
# ==========================================
    def freq_link_rule(m, t):
        return m.delta_f[t] >= m.delta_f_max * (
            1 - sum(m.R_pfr[g, t] for g in m.G) / m.Contingency[t]
        )

    model.FreqLink = Constraint(model.T, rule=freq_link_rule)

    def freq_limit_rule(m, t):
        return m.delta_f[t] <= m.delta_f_max

    model.FreqLimit = Constraint(model.T, rule=freq_limit_rule)


    #def freq_balance_rule(m, t):
    #    return sum(m.R_pfr[g, t] for g in m.G) >= m.Plost[t]
    
    #model.FreqBalance = Constraint(model.T, rule=freq_balance_rule)
#def pfr_droop_rule(m, g, t):
#    return m.R_pfr[g, t] <= m.FreqBias[g] * m.delta_f[t]

#model.PFR_Droop = Constraint(model.G, model.T, rule=pfr_droop_rule)



Solve

In [ ]:
# =========================================
# Solve
# =========================================
from pyomo.opt import SolverStatus, TerminationCondition

solver = SolverFactory('cbc')
results = solver.solve(model, tee=False)

if (results.solver.status == SolverStatus.ok and
    results.solver.termination_condition == TerminationCondition.optimal):
    print("Optimal solution found.")
else:
    print("Model infeasible or not optimal.")


In [ ]:
print("\n===== N-1 LARGEST OUTAGE BACKUP DETAIL =====")

for t in model.T:
    # cari unit thermal online dengan output terbesar
    outage_g = max(
        [g for g in model.G if Type[g] not in ["PV", "WT"]],
        key=lambda g: value(model.P[g, t])
    )

    loss = value(model.P[outage_g, t])

    print(f"\n================ HOUR {t} ================")
    print(f"Outage terbesar: {outage_g} | Loss = {loss:.2f} MW")
    print("Backup units:")

    total_backup = 0

    for g in model.G:
        if g != outage_g and Type[g] not in ["PV", "WT"]:
            u = value(model.u[g, t])
            r = value(model.R[g, t])

            if u > 0.5 and r > 1e-6:
                print(f"  {g} | Reserve = {r:.2f} MW")
                total_backup += r

    margin = total_backup - loss

    print(f"Total Backup = {total_backup:.2f} MW")
    print(f"Margin       = {margin:.2f} MW")
    print("Status       =", "OK" if margin >= -1e-6 else "FAIL")

Results

Blok df_hourly

In [ ]:
# =====================
# Format
# =====================
def fmt(x):
    if abs(x - round(x)) < 1e-6:
        return int(round(x))
    else:
        return round(x, 2)

# =====================
# Build df_hourly
# =====================
from pyomo.environ import value
import pandas as pd

rows = []

for t in model.T:
    row = {
        "Hour": t,
        "Demand": fmt(value(model.Demand[t]))
    }

    total_power = 0
    fuel_cost = 0
    fixed_cost = 0
    startup_cost = 0
    shutdown_cost = 0
    total_reserve = 0
    total_pfr = 0
    reserve_cost = 0

    for g in model.G:
        p = value(model.P[g, t])
        u = value(model.u[g, t])
        y = value(model.y[g, t])
        z = value(model.z[g, t])
        r = value(model.R[g, t])
        r_pfr = value(model.R_pfr[g, t]) if use_pfr else 0

        row[f"{g}_P"] = fmt(p)
        row[f"{g}_U"] = fmt(u)
        row[f"{g}_Y"] = fmt(y)
        row[f"{g}_Z"] = fmt(z)
        row[f"{g}_R"] = fmt(r)
        row[f"{g}_PFR"] = fmt(r_pfr)
        row[f"{g}_Headroom"] = fmt(value(model.Pmax[g]) - p)

        total_power += p
        total_reserve += r
        total_pfr += r_pfr

        fuel_cost += p * value(model.b[g])
        fixed_cost += u * value(model.c[g])
        startup_cost += y * value(model.SU_cost[g])
        shutdown_cost += z * value(model.SD_cost[g])
        reserve_cost += r * value(model.ReserveCost[g])

        #row["Contingency"] = value(model.Contingency[t])
        row["Total_Reserve"] = sum(value(model.R[g, t]) for g in model.G)

        row["Contingency"] = max(
            value(model.P[g, t])
            for g in model.G
            if Type[g] not in ["PV", "WT"]
        )        

        row["Total_Reserve"] = sum(value(model.R[g, t]) for g in model.G)
        row["Reserve_Margin"] = row["Total_Reserve"] - row["Contingency"]

    row["Total_Power"] = fmt(total_power)
    row["Fuel Cost"] = fmt(fuel_cost)
    row["Fixed Cost"] = fmt(fixed_cost)
    row["Startup Cost"] = fmt(startup_cost)
    row["Shutdown Cost"] = fmt(shutdown_cost)
    row["Total Cost"] = fmt(fuel_cost + fixed_cost + startup_cost + shutdown_cost)

    row["Total_Reserve"] = fmt(total_reserve)
    #row["Reserve_Req"] = fmt(value(model.ReserveReq[t]))
    #row["Reserve_Margin"] = fmt(total_reserve - value(model.ReserveReq[t]))

    row["Reserve Cost"] = fmt(reserve_cost)
    row["Total Cost"] = fmt(fuel_cost + fixed_cost + startup_cost + shutdown_cost + reserve_cost)

    if use_pfr:
        row["Total_PFR"] = fmt(total_pfr)
        row["PFR_Req"] = fmt(value(model.PFRReq[t]))
        row["PFR_Margin"] = fmt(total_pfr - value(model.PFRReq[t]))
    else:
        row["Total_PFR"] = 0
        row["PFR_Req"] = 0
        row["PFR_Margin"] = 0

    rows.append(row)

df_hourly = pd.DataFrame(rows)

for g in model.G:
    if Type[g] in ["PV", "WT"]:
        row[f"{g}_Curt"] = value(model.Curt[g, t])

#Debug
print("\n===== CHECK DYNAMIC CONTINGENCY =====")
for t in model.T:
    total_R = sum(value(model.R[g,t]) for g in model.G)
    
    contingency = max(
        value(model.P[g, t])
        for g in model.G
        if Type[g] not in ["PV", "WT"]
    )

    print(f"Hour {t}: Reserve={total_R:.2f}, Contingency={contingency:.2f}")

t = 4

print(f"\n===== DETAIL RESERVE HOUR {t} =====")

contingency = max(
    value(model.P[g, t])
    for g in model.G
    if Type[g] not in ["PV", "WT"]
)

total_R = 0

for g in model.G:
    p = value(model.P[g, t])
    r = value(model.R[g, t])
    pmax = value(model.Pmax[g])
    u = value(model.u[g, t])
    headroom = pmax * u - p

    total_R += r

    print(
        f"{g:>3} | Type={Type[g]:>14} | "
        f"P={p:8.2f} | R={r:8.2f} | "
        f"Headroom={headroom:8.2f} | u={u:.0f}"
    )

print("-" * 60)
print(f"Total Reserve = {total_R:.2f}")
print(f"Contingency   = {contingency:.2f}")
print(f"Margin        = {total_R - contingency:.2f}")    

In [ ]:
# ===============================
# FREQUENCY RESPONSE SIMULATION
# Droop-based PFR
# ===============================

import numpy as np
import matplotlib.pyplot as plt
from pyomo.environ import value

# Parameter sistem
f0 = 50.0
D_frac = 0.01       # 1% load damping per Hz
sim_time = 200
dt = 0.02
time = np.arange(0, sim_time + dt, dt)

# Pilih jam studi
t_uc = 8

# Cari outage terbesar, thermal saja
outage_g = max(
    [g for g in model.G if Type[g] not in ["PV", "WT"]],
    key=lambda g: value(model.P[g, t_uc])
)

loss = value(model.P[outage_g, t_uc])
demand_t = value(model.Demand[t_uc])

# Load damping MW/Hz
D = D_frac * demand_t

# Inertia system setelah outage: unit outage dikeluarkan
Hsys_MWs = sum(
    value(model.H[g]) * value(model.Pmax[g]) * value(model.u[g, t_uc])
    for g in model.G
    if g != outage_g and Type[g] not in ["PV", "WT"]
)

# Unit yang boleh ikut PFR
pfr_units = [
    g for g in model.G
    if g != outage_g
    and Type[g] not in ["PV", "WT"]
    and value(model.u[g, t_uc]) > 0.5
    and value(model.FreeGovernor[g]) == 1
    and value(model.R[g, t_uc]) > 1e-6
]

print("===== FREQUENCY RESPONSE CASE =====")
print("Hour =", t_uc)
print("Outage =", outage_g)
print("Loss =", loss, "MW")
print("Demand =", demand_t, "MW")
print("Hsys =", Hsys_MWs, "MW.s")
print("D =", D, "MW/Hz")
print("PFR units =", pfr_units)

for g in pfr_units:
    print(
        g,
        "| R =", value(model.R[g, t_uc]),
        "| FreqBias =", value(model.FreqBias[g]),
        "| Tg =", value(model.Tg[g])
    )

In [ ]:
# ===============================
# Dynamic simulation
# ===============================

df = 0.0   # deviasi frekuensi dari 50 Hz
freq = []

pfr_response = {g: 0.0 for g in model.G}
pfr_history = {g: [] for g in pfr_units}
total_pfr_history = []

for tau in time:
    total_pfr = 0.0

    for g in pfr_units:
        freq_bias = value(model.FreqBias[g])   # MW/Hz
        tg = value(model.Tg[g])
        r_cap = value(model.R[g, t_uc])        # batas PFR dari reserve

        # Droop-based PFR target
        # df negatif jika frekuensi turun
        target = freq_bias * max(0, -df)

        # dibatasi reserve unit
        target = min(target, r_cap)

        # governor first-order response
        pfr_response[g] += (target - pfr_response[g]) * dt / tg

        total_pfr += pfr_response[g]
        pfr_history[g].append(pfr_response[g])

    # swing equation
    power_imbalance = -loss + total_pfr - D * df

    df_dt = (f0 / (2 * Hsys_MWs)) * power_imbalance
    df += df_dt * dt

    freq.append(f0 + df)
    total_pfr_history.append(total_pfr)

freq = np.array(freq)
total_pfr_history = np.array(total_pfr_history)

rocof_initial = (freq[1] - freq[0]) / dt
nadir = np.min(freq)
t_nadir = time[np.argmin(freq)]
qss = freq[-1]

print("\n===== FREQUENCY RESPONSE RESULT =====")
print(f"Waktu         = {t_uc}")
print(f"Initial RoCoF = {rocof_initial:.4f} Hz/s")
print(f"Nadir         = {nadir:.4f} Hz at {t_nadir:.2f} s")
print(f"QSS Frequency = {qss:.4f} Hz")
print(f"Final PFR     = {total_pfr_history[-1]:.2f} MW")

In [ ]:
print("Outage =", outage_g)
print("Loss =", loss)
print("PFR units =", pfr_units)

for g in pfr_units:
    print(
        g,
        "R =", value(model.R[g, t_uc]),
        "FreqBias =", value(model.FreqBias[g]),
        "Tg =", value(model.Tg[g])
    )

print("Final PFR =", total_pfr_history[-1])
print("Nadir =", nadir)
print("QSS =", qss)

B_total = sum(value(model.FreqBias[g]) for g in pfr_units)
f_qss_est = f0 - loss / (B_total + D)

print("B_total =", B_total)
print("Estimated QSS =", f_qss_est)

Tabel A: Dispatch

In [ ]:
print("\n========== TABEL A: DISPATCH ==========")
print("Keterangan: Output daya tiap unit dan keseimbangan sistem\n")

dispatch_cols = ["Hour", "Demand"] + [f"{g}_P" for g in model.G] + ["Total_Power"]
print(df_hourly[dispatch_cols].to_string(index=False))

Tabel B: Status Unit

In [ ]:
print("\n========== TABEL B: STATUS UNIT ==========")
print("Keterangan: Status ON/OFF unit (1 = ON, 0 = OFF)\n")

status_cols = ["Hour"] + [f"{g}_U" for g in model.G]
print(df_hourly[status_cols].to_string(index=False))

Tabel C: Reserve

In [ ]:
print("\n========== TABEL C: RESERVE ==========")
print("Keterangan: Cadangan daya (spinning reserve) dan margin sistem\n")

reserve_cols = (["Hour", "Contingency"] + [f"{g}_R" for g in model.G] + ["Total_Reserve"])
print(df_hourly[reserve_cols].to_string(index=False))

Tabel D: Biaya

In [ ]:
print("\n========== TABEL D: BIAYA ==========")
print("Keterangan: Komponen biaya pembangkitan per jam\n")

cost_cols = ["Hour", "Fuel Cost", "Fixed Cost", "Startup Cost", "Shutdown Cost", "Total Cost"]
print(df_hourly[cost_cols].to_string(index=False))

Tabel E: Ramping

In [ ]:
print("\n========== TABEL E: RAMPING ==========")
print("Keterangan: Perubahan daya antar jam (ΔP)\n")

ramp_rows = []

for t in model.T:
    ramp_row = {"Hour": t}
    if t == 1:
        for g in model.G:
            ramp_row[f"Δ{g}"] = "-"
    else:
        for g in model.G:
            dp = value(model.P[g, t]) - value(model.P[g, t-1])
            ramp_row[f"Δ{g}"] = fmt(dp)

    ramp_rows.append(ramp_row)

df_ramp = pd.DataFrame(ramp_rows)
print(df_ramp.to_string(index=False))

Tabel F: System Summary

In [ ]:
print("\n========== TABEL F: SYSTEM SUMMARY ==========")
print("Keterangan: Ringkasan performa sistem\n")

peak_demand = max(value(model.Demand[t]) for t in model.T)
total_cost_system = value(model.Obj)

summary_rows = [
    ["Peak Demand", fmt(peak_demand)],
    ["Total System Cost", fmt(total_cost_system)],
 #   ["Minimum Reserve Margin", fmt(df_hourly["Reserve_Margin"].min())],
]

total_startup = sum(value(model.y[g, t]) for g in model.G for t in model.T)
total_shutdown = sum(value(model.z[g, t]) for g in model.G for t in model.T)

summary_rows.append(["Total Startup Events", fmt(total_startup)])
summary_rows.append(["Total Shutdown Events", fmt(total_shutdown)])

df_sys_summary = pd.DataFrame(summary_rows, columns=["Item", "Value"])
print(df_sys_summary.to_string(index=False))

Tabel G: PFR

In [ ]:
print("\n========== TABEL G: PFR ==========")
print("Keterangan: Primary Frequency Reserve per unit dan margin sistem\n")

pfr_cols = ["Hour", "PFR_Req"] + [f"{g}_PFR" for g in model.G] + ["Total_PFR", "PFR_Margin"]
print(df_hourly[pfr_cols].to_string(index=False))

Snapshot jam terpilih

In [ ]:
#selected_hour = t_uc

df_t = df_hourly[df_hourly["Hour"] == t_uc].copy()

print("\n========== FREQUENCY BLOCK 1: SNAPSHOT ==========")
print("Snapshot jam =", t_uc)
print(df_t.to_string(index=False))

Tentukan unit trip (N-1)

In [ ]:
gen_list = list(model.G)

dispatch_at_t = {}

for g in gen_list:
    p = float(df_t[f"{g}_P"].iloc[0])
    u = float(df_t[f"{g}_U"].iloc[0])

    if u > 0.5 and p > 0:
        dispatch_at_t[g] = p

trip_unit = max(dispatch_at_t, key=dispatch_at_t.get)
P_loss = float(dispatch_at_t[trip_unit])

total_reserve = sum(float(df_t[f"{g}_R"].iloc[0]) for g in gen_list)

total_pfr = 0.0
for g in gen_list:
    if g != trip_unit:
        total_pfr += float(df_t[f"{g}_PFR"].iloc[0])

print("\n========== FREQUENCY BLOCK 2: CONTINGENCY ==========")
print("Dispatch unit ON =", dispatch_at_t)
print("Unit trip =", trip_unit)
print("P_loss =", P_loss, "MW")
print("Total Reserve =", total_reserve, "MW")
print("Total PFR =", total_pfr, "MW")

Simulasi frekuensi satu jam

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

f0 = 50.0
H_sys = 5.0
tau_gov = 3.0

t = np.linspace(0, 60, 1000)

P_response = total_pfr * (1 - np.exp(-t / tau_gov))
delta_f = (P_loss - P_response) / (2 * H_sys * 100)
freq_response = f0 - delta_f

f_nadir = np.min(freq_response)
f_ss = freq_response[-1]

print("\n========== FREQUENCY BLOCK 3: SINGLE-HOUR RESPONSE ==========")
print("Frequency nadir =", round(f_nadir, 4), "Hz")
print("Steady-state frequency =", round(f_ss, 4), "Hz")

f_min = 49.5
if f_nadir >= f_min:
    print("Status: AMAN")
else:
    print("Status: TIDAK AMAN")

plt.figure()
plt.plot(t, freq_response)
plt.axhline(50, linestyle='--')
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.title(f"Frequency Response after Generator Trip (Hour {t_uc})")
plt.grid()
plt.show()

Rekap Frekuensi Semua Jam

In [ ]:
summary = []

test_hours = list(range(1, 25))

for th in test_hours:
    df_t = df_hourly[df_hourly["Hour"] == th].copy()

    dispatch = {}
    for g in gen_list:
        p = float(df_t[f"{g}_P"].iloc[0])
        u = float(df_t[f"{g}_U"].iloc[0])
        if u > 0.5 and p > 0:
            dispatch[g] = p

    trip_unit = max(dispatch, key=dispatch.get)
    P_loss = dispatch[trip_unit]

    total_reserve = sum(float(df_t[f"{g}_R"].iloc[0]) for g in gen_list)

    total_pfr = 0.0
    for g in gen_list:
        if g != trip_unit:
            total_pfr += float(df_t[f"{g}_PFR"].iloc[0])

    f0 = 50.0
    H_sys = 5.0
    tau_gov = 3.0

    t = np.linspace(0, 60, 1000)

    P_response = total_pfr * (1 - np.exp(-t / tau_gov))
    delta_f = (P_loss - P_response) / (2 * H_sys * 100)
    freq_response = f0 - delta_f

    f_nadir = np.min(freq_response)
    f_ss = freq_response[-1]

    net_deficit = P_loss - total_pfr
    pfr_ratio = total_pfr / P_loss if P_loss > 0 else 0

    summary.append([
        th,
        trip_unit,
        round(P_loss, 2),
        round(total_reserve, 2),
        round(total_pfr, 2),
        round(net_deficit, 2),
        round(pfr_ratio, 3) * 100,
        round(f_nadir, 4),
        round(f_ss, 4)
    ])

df_freq_summary = pd.DataFrame(summary, columns=[
    "t",
    "Unit Trip",
    "Ploss (MW)",
    "Total Reserve (MW)",
    "Total PFR (MW)",
    "Net Deficit (MW)",
    "PFR Ratio (%)",
    "f_nadir (Hz)",
    "f_steady (Hz)"
])

print("\n========== FREQUENCY BLOCK 4: ALL-HOUR SUMMARY ==========")
print(df_freq_summary.to_string(index=False))

Grafik Frekuensi

Grafik Net Deficit vs f nadir

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.scatter(df_freq_summary["Net Deficit (MW)"], df_freq_summary["f_nadir (Hz)"])

plt.xlabel("Net Deficit (MW)")
plt.ylabel("Frequency Nadir (Hz)")
plt.title("Net Deficit vs Frequency Nadir")
plt.grid()

plt.show()

Grafik PFR Ratio vs f_nadir

In [ ]:
plt.figure()
plt.scatter(df_freq_summary["PFR Ratio (%)"], df_freq_summary["f_nadir (Hz)"])

plt.xlabel("PFR Ratio (%)")
plt.ylabel("Frequency Nadir (Hz)")
plt.title("PFR Ratio vs Frequency Nadir")
plt.grid()

plt.show()

Grafik per jam

In [ ]:
fig, ax1 = plt.subplots()

ax1.set_xlabel("Hour")
ax1.set_ylabel("f_nadir (Hz)")
ax1.plot(df_freq_summary["t"], df_freq_summary["f_nadir (Hz)"])
ax1.grid()

ax2 = ax1.twinx()
ax2.set_ylabel("Ploss (MW)")
ax2.plot(df_freq_summary["t"], df_freq_summary["Ploss (MW)"], linestyle='--')

plt.title("Trend per Hour")
plt.show()

Grafik

In [ ]:
import matplotlib.pyplot as plt
# Mapping nama unit untuk tampilan
name_map = {
    "G1": "Coal",
    "G2": "Gas",
    "G3": "Diesel",
    "G4": "Combined-Cycle",
    "G5": "Gas Machine",
    "G6": "PV",
    "G7": "WT"
}

# Label Legend
plot_labels = [name_map.get(g, g) for g in model.G]

# ==============================================
# 1. DEMAND, RESERVE, PFR, PV/WT POTENTIAL
# ==============================================

hours = list(model.T)

demand = [value(model.Demand[t]) for t in model.T]

total_reserve = [
    sum(value(model.R[g, t]) for g in model.G)
    for t in model.T
]

reserve_req = [
    max(
        value(model.P[g, t])
        for g in model.G
        if Type[g] not in ["PV", "WT"]
    )
    for t in model.T
]

if use_pfr:
    total_pfr = [
        sum(value(model.R_pfr[g, t]) for g in model.G)
        for t in model.T
    ]
    pfr_req = [
        value(model.PFRReq[t])
        for t in model.T
    ]
else:
    total_pfr = [0 for t in model.T]
    pfr_req = [0 for t in model.T]

pv_potential = [
    sum(value(model.RenewPotential[g, t]) for g in model.G if Type[g] == "PV")
    for t in model.T
]

wt_potential = [
    sum(value(model.RenewPotential[g, t]) for g in model.G if Type[g] == "WT")
    for t in model.T
]

plt.figure(figsize=(11, 5))
plt.plot(hours, demand, label="Demand")
plt.plot(hours, total_reserve, label="Total Reserve")
#plt.plot(hours, reserve_req, linestyle="--", label="Reserve Requirement")
#plt.plot(hours, total_pfr, label="Total PFR")
#plt.plot(hours, pfr_req, linestyle="--", label="PFR Requirement")
plt.plot(hours, pv_potential, label="PV Potential")
plt.plot(hours, wt_potential, label="WT Potential")

plt.xlabel("Hour")
plt.ylabel("MW")
plt.title("Demand, Reserve, and Renewable Potential")
#plt.legend()
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.grid()
plt.show()


In [ ]:
# ===============================
# 2. DISPATCH PER UNIT (STACKED)
# ===============================
gen_cols = [f"{g}_P" for g in model.G]

plt.figure()
plt.stackplot(
    df_hourly["Hour"],
    *[df_hourly[col] for col in gen_cols],
    labels=plot_labels
)

# Overlay demand
plt.plot(df_hourly["Hour"], df_hourly["Demand"], linewidth=2, label="Demand")

plt.xlabel("Hour")
plt.ylabel("MW")
plt.title("Generation Dispatch (Stacked)")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# ===============================
# 3. STATUS UNIT (ON/OFF)
# ===============================
import numpy as np
from matplotlib.colors import ListedColormap
from pyomo.environ import value

gen_status_matrix = []

for g in model.G:
    gen_status_matrix.append([
        1 if value(model.P[g, t]) > 1e-4 else 0
        for t in model.T
    ])

# matrix status (0/1)
matrix = np.array(gen_status_matrix)

# colormap: 0 = putih, 1 = biru (bisa diganti)
cmap = ListedColormap(["white", "#4C72B0"])  # biru soft

plt.figure(figsize=(10, 4))

plt.imshow(
    matrix,
    aspect="auto",
    interpolation="nearest",
    cmap=cmap,
    vmin=0, vmax=1
)

plt.xticks(range(len(list(model.T))), list(model.T))
plt.yticks(range(len(list(model.G))), [name_map.get(g, g) for g in model.G])

plt.xlabel("Hour")
plt.ylabel("Generator")
plt.title("Generator Operating Status")

# colorbar khusus biner
cbar = plt.colorbar(ticks=[0, 1])
cbar.ax.set_yticklabels(["OFF", "ON"])

plt.show()

In [ ]:
# ===============================
# 4. TOTAL COST PER HOUR
# ===============================
plt.figure()
plt.plot(df_hourly["Hour"], df_hourly["Total Cost"], label="Total Cost")

plt.xlabel("Hour")
plt.ylabel("Cost")
plt.title("Hourly Total Cost")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# ===============================
# 5. CURTAILMENT
# ===============================
for g in model.G:
    if Type[g] in ["PV", "WT"]:
        potential = [value(model.RenewPotential[g, t]) for t in model.T]
        dispatch  = [value(model.P[g, t]) for t in model.T]
        #curt      = [value(model.Curt[g, t]) for t in model.T]
        pmax_g    = value(model.Pmax[g])

        plt.figure(figsize=(10,4))
        plt.plot(model.T, potential, label="Potential")
        plt.plot(model.T, dispatch, label="Dispatch")
        #plt.plot(model.T, curt, linestyle="--", label="Curtailment")

        # Kapasitas maksimum terpasang
        plt.axhline(
            y=pmax_g,
            linestyle=":",
            linewidth=2,
            label=f"Pmax = {pmax_g} MW"
        )

        plt.title(f"{g} Curtailment")
        plt.xlabel("Hour")
        plt.ylabel("MW")
        plt.legend(loc="center left", bbox_to_anchor=(1,0.5))
        plt.grid()
        plt.tight_layout()
        plt.show()

In [ ]:
# ===============================
# PLOT FREKUENSI + ANOTASI
# ===============================

# waktu nadir
t_nadir = time[np.argmin(freq)]

# QSS
qss = freq[-1]

# estimasi waktu steady state:
# titik pertama setelah nadir ketika frekuensi sudah dekat QSS
tol_qss = 0.01   # toleransi 0.01 Hz
idx_nadir = np.argmin(freq)

idx_qss = len(freq) - 1
for i in range(idx_nadir, len(freq)):
    if abs(freq[i] - qss) <= tol_qss:
        idx_qss = i
        break

t_qss = time[idx_qss]

# RoCoF
rocof_series = np.diff(freq) / dt
rocof_initial = rocof_series[0]
rocof_min = np.min(rocof_series)

print("\n===== HASIL ANALISIS FREKUENSI =====")
print(f"Hour              = {t_uc}")
print(f"Outage            = {outage_g}")
print(f"Loss              = {loss:.2f} MW")
print(f"Nadir             = {nadir:.3f} Hz")
print(f"Time of Nadir     = {t_nadir:.2f} s")
print(f"QSS               = {qss:.3f} Hz")
print(f"Time of QSS       = {t_qss:.2f} s")
print(f"Initial RoCoF     = {rocof_initial:.3f} Hz/s")
print(f"Minimum RoCoF     = {rocof_min:.3f} Hz/s")

plt.figure(figsize=(11, 5))

plt.plot(time, freq, linewidth=2, label="Frequency Response")

# garis frekuensi
plt.axhline(f0, linestyle="--", label=f"Nominal Frequency = {f0:.2f} Hz")
plt.axhline(nadir, linestyle=":", label=f"Nadir = {nadir:.3f} Hz")
plt.axhline(qss, linestyle="-.", label=f"Steady State Frequency = {qss:.3f} Hz")

# garis waktu
plt.axvline(t_nadir, linestyle=":", label=f"Time of Nadir = {t_nadir:.2f} s")
plt.axvline(t_qss, linestyle="-.", label=f"Time of QSS = {t_qss:.2f} s")

# titik nadir dan qss
plt.scatter(t_nadir, nadir, zorder=5)
plt.scatter(t_qss, qss, zorder=5)

# anotasi nadir
plt.annotate(
    f"Nadir\n{nadir:.3f} Hz\n@ {t_nadir:.2f} s",
    xy=(t_nadir, nadir),
    xytext=(t_nadir + 8, nadir + 0.8),
    arrowprops=dict(arrowstyle="->"),
    bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8)
)

# anotasi QSS
plt.annotate(
    f"QSS\n{qss:.3f} Hz\n@ {t_qss:.2f} s",
    xy=(t_qss, qss),
    xytext=(t_qss + 8, qss + 0.5),
    arrowprops=dict(arrowstyle="->"),
    bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8)
)

# tampilkan RoCoF di plot
plt.text(
    0.02, 0.05,
    f"Initial RoCoF = {rocof_initial:.3f} Hz/s\nMinimum RoCoF = {rocof_min:.3f} Hz/s",
    transform=plt.gca().transAxes,
    bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8)
)

plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.title(f"Frequency Response | Hour {t_uc} | Outage {outage_g}")
plt.legend(loc="best")
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
# =======================================
plt.figure(figsize=(10, 4))

for g in pfr_units:
    plt.plot(time, pfr_history[g], label=f"{g} PFR")

plt.plot(time, total_pfr_history, linewidth=2, label="Total PFR")
plt.axhline(loss, linestyle="--", label=f"Loss = {loss:.1f} MW")

plt.xlabel("Time (s)")
plt.ylabel("PFR Response (MW)")
plt.title(f"PFR Response | Hour {t_uc} | Outage {outage_g}")
plt.legend()
plt.grid()
plt.show()